# Simple Chat Chain

# Basic Notions

## 1. LLM

In [1]:
# GROQ LLM setup

from langchain_groq import ChatGroq
from decouple import Config, RepositoryEnv
env_config = Config(RepositoryEnv("./.env"))

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    # model="llama-3.1-8b-instant",
    temperature=1.0,
    max_retries=2,
    api_key=env_config("GROQ_API_KEY"),
)

/home/niuniu/Documents/tmp/test/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Prompt Template

In [2]:
# static prompt

query = "What is the capital of France?"

res = llm.invoke(query)
print(res)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.006957915, 'completion_tokens_details': None, 'prompt_time': 0.006463352, 'prompt_tokens_details': None, 'queue_time': 0.036677891, 'total_time': 0.013421267}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_d42c28f9ce', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9906-437c-75a0-ac4b-5e0557ebe32c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


In [ ]:
# template prompt
from langchain_core.prompts import ChatPromptTemplate

# simple string template
template = ChatPromptTemplate.from_template("You are a helpful assistant. {input}")

prompt = template.format_messages(input="What is the capital of France?")
print("from_template: ", prompt)


# list template
template = ChatPromptTemplate.from_messages(["You are a helpful assistant. What is the capital of {country}?"])
prompt = template.format_messages(
    country="France"
)
print("from_messages: ", prompt)

from_template:  [HumanMessage(content='You are a helpful assistant. What is the capital of France?', additional_kwargs={}, response_metadata={})]
from_messages:  [HumanMessage(content='You are a helpful assistant. What is the capital of France?', additional_kwargs={}, response_metadata={})]


In [ ]:
# tuple prompt
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{input}")
])

prompt = template.format_messages(input="What is the capital of France?")
print(prompt)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})]


In [35]:
# placeholder
from langchain_core.prompts import ChatPromptTemplate

# tuple prompt with placeholder
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{conversation}"),
    ("user", "{input}")
])

inputs = {
    "input": "What is the capital of France?",
    "conversation": [
        ("human", "Hi, I am John."),
        ("ai", "Hello John! How can I help you?")
    ]
}

prompt = template.format_messages(**inputs)
print("tuple: ", prompt)


# MessagePlaceholder object
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="conversation"),
    ("user", "{input}")
])

prompt = template.format_messages(**inputs)
print("MessagePlaceholder: ", prompt)

tuple:  [SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi, I am John.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello John! How can I help you?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})]
MessagePlaceholder:  [SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi, I am John.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello John! How can I help you?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})]


## 3. Chain

In [ ]:
# chaining with whole input
chain = template | llm
res = chain.invoke(inputs)
print("chain result: ", res)

chain result:  content="The capital of France is Paris. Is there anything else you'd like to know about France or would you like help with something else?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 73, 'total_tokens': 101, 'completion_time': 0.055510939, 'completion_tokens_details': None, 'prompt_time': 0.003728442, 'prompt_tokens_details': None, 'queue_time': 0.036329762, 'total_time': 0.059239381}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9983-7e4d-7f82-bb10-7e7d5ccc8cb5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 73, 'output_tokens': 28, 'total_tokens': 101}


In [37]:
print(type(res))

<class 'langchain_core.messages.ai.AIMessage'>


In [67]:
# chaining with input runner
from langchain_core.runnables import RunnablePassthrough
chain_runner = (
    {
        "conversation": lambda x: [
            ("human", "Hi, I am John."),
            ("ai", "Hello John! How can I help you?")
        ],
        "input": RunnablePassthrough(),
    } 
    | template
    | llm
)
res = chain_runner.invoke("What is the capital of France?")
print("chain with runner result: ", res)

chain with runner result:  content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 73, 'total_tokens': 81, 'completion_time': 0.011950215, 'completion_tokens_details': None, 'prompt_time': 0.004456714, 'prompt_tokens_details': None, 'queue_time': 0.106786181, 'total_time': 0.016406929}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ea7ae-61b5-7853-8b00-d9a92fd1f0b9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 73, 'output_tokens': 8, 'total_tokens': 81}


## 4. Output Parser

In [49]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage

parser = StrOutputParser()
message = AIMessage(content="The capital of France is Paris. aklzejfdaza. 25/8/2023 *-ç''&é")
parsed_res = parser.parse(message)
print("parsed result: ", parsed_res)

parsed result:  content="The capital of France is Paris. aklzejfdaza. 25/8/2023 *-ç''&é" additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


In [50]:
# chaining

chain = template | llm | parser
res = chain.invoke(inputs)
print("chain with parser result: ", res)

chain with parser result:  The capital of France is Paris. Is there anything else you'd like to know about France or would you like to ask about something else?


## 5. Structured Output

In [54]:
from pydantic import BaseModel, Field
class ResponseModel(BaseModel):
    country: str = Field(..., description="The mentioned country.")
    capital: str = Field(..., description="The capital of the country.")

structured_llm = llm.with_structured_output(ResponseModel)
chain = template | structured_llm
res = chain.invoke(inputs)
print("chain with structured model result: ", res)

chain with structured model result:  country='France' capital='Paris'


# Chat Chain

In [104]:
# manually manage the conversation history

from langchain_groq import ChatGroq
from decouple import Config, RepositoryEnv
env_config = Config(RepositoryEnv("./.env"))
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


# 1. llm setup
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1.0,
    max_retries=2,
    api_key=env_config("GROQ_API_KEY"),
)

# 2. prompt template setup
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="conversation"),
    ("user", "{input}")
])

# 3. base chain
base_chain = template | llm

# 4. Create a list to register the conversation history
conversation_history = []

# 5. chain with memory
def chain_with_memory(input: str) -> str:
    # append the new input to the conversation history
    conversation_history.append(("human", input))
    
    # format the prompt with the updated conversation history
    prompt = {"input": input, "conversation": conversation_history}
    
    # get the response from the llm
    response = base_chain.invoke(prompt).content
    
    # append the response to the conversation history
    conversation_history.append(("ai", response))
    
    return response

In [105]:
# to use
chain_with_memory("My name is John. What is the capital of France?")
print("chain with memory result: ", conversation_history[-1][1])

chain_with_memory("What is my name?")
print("chain with memory result: ", conversation_history[-1][1])

chain with memory result:  Hello John, the capital of France is Paris.
chain with memory result:  Your name is John.


In [99]:
# use RunnableWithMessageHistory to manage the conversation history

from langchain_groq import ChatGroq
from decouple import Config, RepositoryEnv
env_config = Config(RepositoryEnv("./.env"))
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory


# 1. llm setup
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1.0,
    max_retries=2,
    api_key=env_config("GROQ_API_KEY"),
)

# 2. prompt template setup
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="conversation"),
    ("user", "{input}")
])

# 3. base chain
# base_chain = template | llm
base_chain = template | llm


# 4. Create a single history instance
history = InMemoryChatMessageHistory()

# 4. chain with memory
chain_with_memory = RunnableWithMessageHistory(
    base_chain, 
    lambda x: history,
    input_messages_key="input",
    history_messages_key="conversation",
)

In [101]:
config = {
    "configurable": {
        "session_id": "user1"
    }
}
res = chain_with_memory.invoke({"input": "What is the capital of France?"}, config=config)
print("chain with memory result: ", res)
res = chain_with_memory.invoke({"input": "Which country were we talking about?"}, config=config)
print("chain with memory result: ", res)

chain with memory result:  content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 72, 'total_tokens': 80, 'completion_time': 0.024969355, 'completion_tokens_details': None, 'prompt_time': 0.003687882, 'prompt_tokens_details': None, 'queue_time': 0.106153674, 'total_time': 0.028657237}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ea7c5-1e7b-7941-8a6d-4d47e2d94046-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 72, 'output_tokens': 8, 'total_tokens': 80}
chain with memory result:  content='We were talking about France.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 96, 'total_tokens': 103, 'completion_time': 0.006957137, 'completion_tokens_details': None, 'prompt_time': 0.038554774, 'pr